# Bench `parse_pdf` sur le corpus réel

Itère `parse_pdf` sur tous les PDFs de `data/` et produit un tableau récapitulatif :

| Colonne | Description |
|---|---|
| `path`                  | Chemin relatif depuis le repo root |
| `corpus`                | Sous-dossier de `data/` (MRH, annuel_reports, insurance, …) |
| `n_pages`               | Nombre total de pages du PDF |
| `source_tool`           | Outil détecté via metadata (Adobe InDesign / CamScanner / …) |
| `source_category`       | `word_native` / `design_tool` / `scanned` / `other` |
| `content_type`          | Verdict global : `native` / `scanned_no_text` / `scanned_with_ocr` / `mixed` / `empty` |
| `recommended_strategy`  | Routage pour le pipeline aval |
| `n_native`, `n_scanned`, `n_mixed`, `n_empty` | Distribution des `page_type` |
| `pages_needing_ocr`     | Nombre de pages à OCR-iser |
| `parse_seconds`         | Temps d'exécution `parse_pdf` |
| `error`                 | Message d'erreur si échec |

Le tableau est exporté en `bench_parse_pdf_results.csv` à la racine du repo.

In [ ]:
import sys
!{sys.executable} -m pip install -q -e ..

In [ ]:
import time
from pathlib import Path
import pandas as pd
from docpipeline.parsing.pdf.parse_pdf import parse_pdf

REPO_ROOT = Path('..').resolve()
DATA_DIR  = REPO_ROOT / 'data'
PDFS = sorted(DATA_DIR.rglob('*.pdf'))
print(f'Found {len(PDFS)} PDFs')

In [ ]:
rows = []
for i, pdf in enumerate(PDFS, 1):
    rel = pdf.relative_to(REPO_ROOT).as_posix()
    corpus = pdf.relative_to(DATA_DIR).parts[0] if pdf.is_relative_to(DATA_DIR) else ''
    print(f'[{i}/{len(PDFS)}] {rel}', flush=True)
    row = {'path': rel, 'corpus': corpus, 'filename': pdf.name}
    t0 = time.perf_counter()
    try:
        result = parse_pdf(pdf)
        s = result['doc_summary']
        ptc = s.get('page_type_counts', {})
        row.update({
            'n_pages':              s['n_pages'],
            'source_tool':          s['source_tool'],
            'source_category':      s['source_category'],
            'content_type':         s['content_type'],
            'recommended_strategy': s['recommended_strategy'],
            'n_native':             ptc.get('native', 0) + ptc.get('native_with_image', 0),
            'n_scanned':            ptc.get('scanned', 0) + ptc.get('scanned_ocr_good', 0) + ptc.get('scanned_ocr_bad', 0),
            'n_mixed':              ptc.get('mixed', 0),
            'n_empty':              ptc.get('empty', 0),
            'pages_needing_ocr':    len(s.get('pages_needing_ocr', [])),
            'ocr_quality':          s.get('ocr_quality'),
            'parse_seconds':        round(time.perf_counter() - t0, 2),
            'error':                None,
        })
    except Exception as e:
        row.update({'parse_seconds': round(time.perf_counter() - t0, 2), 'error': f'{type(e).__name__}: {e}'})
    rows.append(row)

df = pd.DataFrame(rows)
df.to_csv(REPO_ROOT / 'bench_parse_pdf_results.csv', index=False, encoding='utf-8')
print(f'\n✓ Saved {len(df)} rows to bench_parse_pdf_results.csv')

In [ ]:
# Vue d'ensemble
print('=== Erreurs ===')
errors = df[df['error'].notna()]
print(f'{len(errors)} / {len(df)} PDFs en erreur')
if not errors.empty:
    print(errors[['path', 'error']].to_string(index=False))

print('\n=== Distribution des content_type ===')
print(df['content_type'].value_counts(dropna=False).to_string())

print('\n=== Distribution des source_tool ===')
print(df['source_tool'].value_counts(dropna=False).head(15).to_string())

print('\n=== Distribution des recommended_strategy ===')
print(df['recommended_strategy'].value_counts(dropna=False).to_string())

print('\n=== Pages totales / pages OCR ===')
print(f"Total pages : {int(df['n_pages'].sum() or 0)}")
print(f"Pages needing OCR : {int(df['pages_needing_ocr'].sum() or 0)}")
print(f"Temps total : {df['parse_seconds'].sum():.1f}s")